In [1]:
import sys
sys.path.append('..')

import torch
import torch.nn  as nn

from bound_propagation import BoundModelFactory, HyperRectangle

from torchsummary import summary

In [2]:
class Network(nn.Sequential):
    def __init__(self, in_size):

        super().__init__(
            nn.Linear(in_size, 16),
            nn.Tanh(),
            nn.Linear(16, 16),
            nn.Tanh(),
            nn.Linear(16, n_classes)
        )

#----------------------------------------------------
in_ch = 1
in_size = 5
n_classes = 2

net = Network(in_size)

factory = BoundModelFactory()
net = factory.build(net)

summary(net, (in_ch, in_size, in_size))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1             [-1, 1, 5, 16]              96
            Linear-2             [-1, 1, 5, 16]              96
              Tanh-3             [-1, 1, 5, 16]               0
              Tanh-4             [-1, 1, 5, 16]               0
            Linear-5             [-1, 1, 5, 16]             272
            Linear-6             [-1, 1, 5, 16]             272
              Tanh-7             [-1, 1, 5, 16]               0
              Tanh-8             [-1, 1, 5, 16]               0
            Linear-9              [-1, 1, 5, 2]              34
           Linear-10              [-1, 1, 5, 2]              34
Total params: 804
Trainable params: 804
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.01
Params size (MB): 0.00
Estimated Total Siz

### Define Input

In [3]:
x = torch.rand(10, in_size)
print(x.shape)
print(x)

torch.Size([10, 5])
tensor([[0.8544, 0.6599, 0.4136, 0.1579, 0.0474],
        [0.6773, 0.8260, 0.1512, 0.2239, 0.0395],
        [0.8328, 0.0822, 0.8268, 0.2008, 0.2806],
        [0.9706, 0.4917, 0.0562, 0.3496, 0.7603],
        [0.7677, 0.1692, 0.9645, 0.8486, 0.8890],
        [0.1426, 0.9423, 0.1977, 0.3585, 0.0093],
        [0.6641, 0.5454, 0.4920, 0.1065, 0.7352],
        [0.2948, 0.5041, 0.9888, 0.1701, 0.8614],
        [0.8768, 0.8830, 0.4439, 0.8217, 0.6212],
        [0.6792, 0.4430, 0.4208, 0.3921, 0.3483]])


### IBP

In [9]:
epsilon = 0.1
input_bounds = HyperRectangle.from_eps(x, epsilon)

print(len(input_bounds))

ibp_bounds = net.ibp(input_bounds) # defined in the general.py
print(len(ibp_bounds))
print(type(ibp_bounds))

10
10
<class 'bound_propagation.bounds.IntervalBounds'>


In [5]:
crown_bounds = net.crown(input_bounds).concretize()
crown_ibp_bounds = net.crown_ibp(input_bounds).concretize()

alpha_crown_bounds = net.crown(input_bounds, alpha=True).concretize()
alpha_crown_ibp_bounds = net.crown_ibp(input_bounds, alpha=True).concretize()